In [83]:
import pandas as pd
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pytz
import requests
import psycopg2

In [84]:
def read_db_credentials(path="data/config.txt"):
    creds = {}
    with open(path, "r") as f:
        for line in f:
            key, value = line.strip().split("=")
            creds[key] = value
    return creds

def connect_to_db(creds):
    return psycopg2.connect(
        host=creds["host"],
        port=creds["port"],
        dbname=creds["database"],
        user=creds["user"],
        password=creds["password"]
    )

In [85]:
def read_db_credentials(path="data/config.txt"):
    creds = {}
    with open(path, "r") as f:
        for line in f:
            key, value = line.strip().split("=")
            creds[key] = value
    return creds

def connect_to_db(creds):
    return psycopg2.connect(
        host=creds["host"],
        port=creds["port"],
        dbname=creds["database"],
        user=creds["user"],
        password=creds["password"]
    )
def format_columns(cols):
    if isinstance(cols, (list, tuple)):
        return ",".join(map(str, cols))
    return str(cols)

def request_json_cell(user_number, data_source):
    creds = read_db_credentials()
    conn = connect_to_db(creds)
    
    query = """
        SELECT raw_json 
        FROM fact_raw_data
        WHERE user_number = %s AND data_source = %s;
    """
    
    cursor = conn.cursor()
    cursor.execute(query, (user_number, data_source))
    
    row = cursor.fetchone()  # nur eine Zelle erwarten
    cursor.close()
    conn.close()
    
    if row:
        return row[0]  # JSON-Zelle als String
    else:
        return None


    


In [86]:
# json_string = request_json_cell(2, "spotify")

# if json_string:
#     data = json_string  # kein json.loads nötig
#     streaming_data = data.get("StreamingHistory_music_0", [])

#     import pandas as pd
#     df = pd.DataFrame(streaming_data)
#     df_filtered = df[["trackName", "artistName"]]
#     display(df_filtered)

In [87]:
import requests
import base64

client_id = '8541e8dfc0fd4a86916d0d98cdb150ad'
client_secret = 'cade2256b5804831a11349f86e6a1784'

import requests
import base64
import time

class SpotifyAuth:
    def __init__(self, client_id, client_secret):
        self.client_id = client_id
        self.client_secret = client_secret
        self.access_token = None
        self.token_expires_at = 0  # Unix timestamp

    def _fetch_token(self):
        auth_str = f"{self.client_id}:{self.client_secret}"
        b64_auth_str = base64.b64encode(auth_str.encode()).decode()

        response = requests.post(
            "https://accounts.spotify.com/api/token",
            data={"grant_type": "client_credentials"},
            headers={"Authorization": f"Basic {b64_auth_str}"}
        )

        if response.status_code == 200:
            token_data = response.json()
            self.access_token = token_data["access_token"]
            # Spotify gibt Gültigkeit in Sekunden an (meist 3600)
            self.token_expires_at = time.time() + token_data.get("expires_in", 3600) - 60
        else:
            raise Exception(f"Fehler beim Abrufen des Tokens: {response.status_code} - {response.text}")


    def get_token(self):
        if not self.access_token or time.time() >= self.token_expires_at:
            self._fetch_token()
        return self.access_token
    

auth = SpotifyAuth(client_id = '8541e8dfc0fd4a86916d0d98cdb150ad',
client_secret = 'cade2256b5804831a11349f86e6a1784')



In [88]:
from collections import Counter
import requests

artist_genre_cache = {}  # Cache für Artist-Genres

def get_weighted_genres(track_query, target_artist):
    if target_artist.lower() in artist_genre_cache:
        return artist_genre_cache[target_artist.lower()]

    access_token = auth.get_token()
    headers = {"Authorization": f"Bearer {access_token}"}
    params = {"q": track_query, "type": "track", "limit": 20}
    response = requests.get("https://api.spotify.com/v1/search", headers=headers, params=params)
    data = response.json()

    genre_counter = Counter()
    total_artist_matches = 0

    for track in data.get("tracks", {}).get("items", []):
        matched_artist = next(
            (artist for artist in track["artists"] if artist["name"].lower() == target_artist.lower()), None
        )
        if matched_artist:
            total_artist_matches += 1
            artist_id = matched_artist["id"]
            artist_data = requests.get(f"https://api.spotify.com/v1/artists/{artist_id}", headers=headers).json()
            genres = artist_data.get("genres", [])
            genre_counter.update(genres)

    if total_artist_matches == 0 or not genre_counter:
        artist_genre_cache[target_artist.lower()] = {}
        return {}

    total_genre_mentions = sum(genre_counter.values())
    normalized_genres = {
        genre: round(count / total_genre_mentions, 3)
        for genre, count in genre_counter.items()
    }

    artist_genre_cache[target_artist.lower()] = normalized_genres
    return normalized_genres




In [89]:

with open("Streaming_History_Audio_2019-2021.json", "r", encoding="utf-8") as f:
    spotify_raw = json.load(f)
streaming_data = spotify_raw
print(streaming_data)

import pandas as pd
df = pd.DataFrame(streaming_data)
# Angenommen deine Zeitspalte heißt z. B. "ts" oder "endTime"
df["ts"] = pd.to_datetime(df["ts"])  # Zeitspalte umwandeln

# Nach Zeit sortieren (neueste zuerst)
df_sorted = df.sort_values(by="ts", ascending=False)

# Letzte 300 Einträge auswählen
df_last_300 = df_sorted.head(500)


[{'ts': '2019-04-03T00:22:20Z', 'platform': 'iOS 11.4 (iPhone9,3)', 'ms_played': 61226, 'conn_country': 'EG', 'ip_addr': '41.40.126.7', 'master_metadata_track_name': 'Bayen Habeit', 'master_metadata_album_artist_name': 'Marshmello', 'master_metadata_album_album_name': 'Bayen Habeit', 'spotify_track_uri': 'spotify:track:29KS7kBSU77PrYGB6RVR5O', 'episode_name': None, 'episode_show_name': None, 'spotify_episode_uri': None, 'audiobook_title': None, 'audiobook_uri': None, 'audiobook_chapter_uri': None, 'audiobook_chapter_title': None, 'reason_start': 'clickrow', 'reason_end': 'endplay', 'shuffle': False, 'skipped': False, 'offline': False, 'offline_timestamp': None, 'incognito_mode': False}, {'ts': '2019-04-03T00:22:23Z', 'platform': 'iOS 11.4 (iPhone9,3)', 'ms_played': 2730, 'conn_country': 'EG', 'ip_addr': '41.40.126.7', 'master_metadata_track_name': 'rockstar (feat. 21 Savage)', 'master_metadata_album_artist_name': 'Post Malone', 'master_metadata_album_album_name': 'beerbongs & bentleys'

In [90]:


df_filtered = df_last_300[['master_metadata_track_name', 'master_metadata_album_artist_name']].copy()

df_filtered["weightedGenres"] = df_filtered.apply(
    lambda row: get_weighted_genres(row['master_metadata_track_name'], row['master_metadata_album_artist_name']),
    axis=1
)

print(df_filtered)
print(df_filtered.columns)

                             master_metadata_track_name  \
4972                    Heaven To Me (feat. Alex Clare)   
4971                              No Sleep (feat. Bonn)   
4970                              No Sleep (feat. Bonn)   
4969                            I Wanna Know - Acoustic   
4968                            I Wanna Know - Acoustic   
...                                                 ...   
4477                                            Rapture   
4476                                   Waves - Acoustic   
4475  Rewrite The Stars (with James Arthur & Anne-Ma...   
4474                                         Let You Go   
4473                                      Lost In Japan   

     master_metadata_album_artist_name  \
4972                        Don Diablo   
4971                     Martin Garrix   
4970                     Martin Garrix   
4969                              NOTD   
4968                              NOTD   
...                                ... 

In [91]:
import pandas as pd
import re


# Regex-basierte Mapping-Regeln (von spezifisch zu allgemein)
REGEX_GENRE_TO_USER_TYPE = {
    # HIPHOP / RAP
    r".*\bhip hop\b.*": "hiphop_head",
    r".*\bgrime\b.*": "hiphop_head",
    r".*\bdrill\b.*": "hiphop_head",
    r".*\brap\b.*": "hiphop_head",
    r".*\burbaine\b.*": "hiphop_head",
    r".*\btrap\b.*": "hiphop_head",
    r".*\bgrime\b.*": "hiphop_head",
    r".*\buk grime\b.*": "hiphop_head",

    # ELECTRONIC
    r".*\bhouse\b.*": "electronic_addict",
    r".*\btrance\b.*": "electronic_addict",
    r".*\btechno\b.*": "electronic_addict",
    r".*\bbig room\b.*": "electronic_addict",
    r".*\bedm\b.*": "electronic_addict",
    r".*\belectronica\b.*": "electronic_addict",
    r".*\bdance\b.*": "electronic_addict",
    r".*\bdrum and bass\b.*": "electronic_addict",
    r".*\bfuture house\b.*": "electronic_addict",
    r".*\bprogressive house\b.*": "electronic_addict",
    r".*\btropical house\b.*": "electronic_addict",

    # POP
    r".*\bpop\b.*": "pop_lover",
    r".*\bsoft pop\b.*": "pop_lover",
    r".*\bart pop\b.*": "pop_lover",
    r".*\bindie pop\b.*": "pop_lover",
    r".*\bmodern pop\b.*": "pop_lover",
    r".*\bmoroccan pop\b.*": "pop_lover",
    r".*\begyptian pop\b.*": "pop_lover",
    r".*\bturkish pop\b.*": "pop_lover",
    r".*\blatin pop\b.*": "pop_lover",
    r".*\bk-pop\b.*": "pop_lover",
    r".*\bt-pop\b.*": "pop_lover",

    # INDIE / ALTERNATIVE
    r".*\bindie rock\b.*": "indie_explorer",
    r".*\balternative rock\b.*": "indie_explorer",
    r".*\bindie\b.*": "indie_explorer",
    r".*\bsinger-songwriter\b.*": "indie_explorer",
    r".*\bitalian singer-songwriter\b.*": "indie_explorer",

    # JAZZ
    r".*\bjazz\b.*": "jazz_purist",
    r".*\bfrench jazz\b.*": "jazz_purist",

    # CLASSICAL / MUSICAL
    r".*\bopera\b.*": "classical_connoisseur",
    r".*\bmusicals\b.*": "classical_connoisseur",
    r".*\bnueva trova\b.*": "classical_connoisseur",

    # ROCK
    r".*\brock\b.*": "rock_head",
    r".*\bgarage rock\b.*": "rock_head",

    # FUNK / SOUL / R&B
    r".*\br&b\b.*": "funk_soul_groover",
    r".*\bfrench r&b\b.*": "funk_soul_groover",
    r".*\bsoul\b.*": "funk_soul_groover",

    # COUNTRY / FOLK
    r".*\bfolk\b.*": "country_heart",
    r".*\bfolk pop\b.*": "country_heart",
    r".*\bmariachi\b.*": "country_heart",
    r".*\btrova\b.*": "country_heart",
    r".*\bbolero\b.*": "country_heart",

    # GLOBAL
    r".*\bafro.*": "global_vibes_fan",
    r".*\bdembow\b.*": "global_vibes_fan",
    r".*\blatin\b.*": "global_vibes_fan",
    r".*\bmoroccan rap\b.*": "global_vibes_fan",
    r".*\bkhaleeji\b.*": "global_vibes_fan",
    r".*\barabic hip hop\b.*": "global_vibes_fan",
    r".*\bvariet[eé] fran[aç]aise\b.*": "global_vibes_fan",
    r".*\bchanson\b.*": "global_vibes_fan",
    r".*\bitalo dance\b.*": "global_vibes_fan",
    r".*\barabesk\b.*": "global_vibes_fan",
    r".*\bra[iï]\b.*": "global_vibes_fan",
}


# Funktion zum Zuordnen via Regex
def map_genre_to_user_type(genre):
    for pattern, user_type in REGEX_GENRE_TO_USER_TYPE.items():
        if re.search(pattern, genre, re.IGNORECASE):
            return user_type
    return "unknown"




In [92]:
rows = []
for _, row in df_filtered.iterrows():
    genres_dict = row.get("weightedGenres", {})
    if isinstance(genres_dict, dict):
        for genre, weight in genres_dict.items():
            rows.append({
                "trackName": row['master_metadata_track_name'],
                "artistName": row['master_metadata_album_artist_name'],
                "genre": genre,
                "weight": weight
            })

df_genres = pd.DataFrame(rows)
print(df_genres)
genre_summary = df_genres.groupby("genre", as_index=False)["weight"].sum().sort_values(by="weight", ascending=False)
genre_summary["user_type"] = genre_summary["genre"].map(GENRE_TO_USER_TYPE)


genre_summary["user_type"] = genre_summary["genre"].apply(map_genre_to_user_type)
print(genre_summary["genre"])
# Gruppieren und Gewicht aufsummieren
result = genre_summary.groupby("user_type", as_index=False)["weight"].sum()
total_weight = result ["weight"].sum()
result["normalized_weight"] = (result["weight"] / total_weight)*100
display(result.sort_values("normalized_weight", ascending=False))

                                             trackName     artistName  \
0                      Heaven To Me (feat. Alex Clare)     Don Diablo   
1                      Heaven To Me (feat. Alex Clare)     Don Diablo   
2                                No Sleep (feat. Bonn)  Martin Garrix   
3                                No Sleep (feat. Bonn)  Martin Garrix   
4                                No Sleep (feat. Bonn)  Martin Garrix   
..                                                 ...            ...   
345                          High On Life (feat. Bonn)  Martin Garrix   
346                          High On Life (feat. Bonn)  Martin Garrix   
347                    Speechless (feat. Erika Sirola)   Robin Schulz   
348                                            Find Me          Sigma   
349  Rewrite The Stars (with James Arthur & Anne-Ma...   James Arthur   

                 genre  weight  
0         future house   0.500  
1                  edm   0.500  
2                  edm  

,user_type,weight,normalized_weight
2,electronic_addict,101.978,47.882391
7,pop_lover,42.750,20.072684
0,classical_connoisseur,27.750,13.029637
4,global_vibes_fan,13.083,6.142946
6,indie_explorer,9.499,4.460127
5,hiphop_head,9.166,4.303771
9,unknown,5.000,2.347682
1,country_heart,2.250,1.056457
3,funk_soul_groover,1.000,0.469536
8,rock_head,0.500,0.234768


In [93]:
import pandas as pd
from datetime import datetime
#print(df)
# Zeitstempel umwandeln
df['ts'] = pd.to_datetime(df['ts'])

# Zeitstempel umwandeln
df['ts'] = pd.to_datetime(df['ts'])

# Minuten berechnen
df['minutes'] = df['ms_played'] / 1000 / 60

# Datum extrahieren
df['date'] = df['ts'].dt.date

# Wochentag ermitteln: Montag = 0, Sonntag = 6
df['weekday'] = df['ts'].dt.weekday
df['is_weekend'] = df['weekday'] >= 5  # Samstag=5, Sonntag=6 → True

# Minuten pro Tag + Wochenende-Markierung
daily_minutes = df.groupby(['date', 'is_weekend'])['minutes'].sum().reset_index()

# Mittelwert berechnen
mean_per_group = daily_minutes.groupby('is_weekend')['minutes'].mean().reset_index()

# Lesbare Labels
mean_per_group['day_type'] = mean_per_group['is_weekend'].map({True: 'Weekend', False: 'Weekday'})
mean_per_group = mean_per_group[['day_type', 'minutes']]

print(daily_minutes)
print(mean_per_group)

# Tageszeit-Funktion
def get_day_part(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "midday"
    elif 17 <= hour < 23:
        return "evening"
    else:
        return "night"

# Attribute berechnen
df['date'] = df['ts'].dt.date
df['weekday'] = df['ts'].dt.weekday  # Monday = 0
df['is_weekend'] = df['weekday'] >= 5
df['day_part'] = df['ts'].dt.hour.apply(get_day_part)

# Aggregation: Minuten pro Tagtyp und Tageszeit
df['day_type'] = df['is_weekend'].map({True: 'Weekend', False: 'Weekday'})
print(df)
# Gruppenbasierte Mittelwerte berechnen
grouped = df.groupby(['date','day_type', 'day_part'])['minutes'].sum().reset_index().groupby(['day_type', 'day_part'])['minutes'].mean().reset_index()


print(grouped)


           date  is_weekend     minutes
0    2019-04-03       False   88.460850
1    2019-04-04       False   96.349233
2    2019-04-05       False   30.938133
3    2019-04-07        True   18.558467
4    2019-04-08       False   17.535867
..          ...         ...         ...
123  2021-03-04       False   27.546800
124  2021-03-05       False   97.539267
125  2021-03-06        True  122.909883
126  2021-03-26       False    8.410917
127  2021-04-14       False   38.030167

[128 rows x 3 columns]
  day_type    minutes
0  Weekday  69.693079
1  Weekend  57.925475
                            ts                                      platform  \
0    2019-04-03 00:22:20+00:00                          iOS 11.4 (iPhone9,3)   
1    2019-04-03 00:22:23+00:00                          iOS 11.4 (iPhone9,3)   
2    2019-04-03 00:25:23+00:00                          iOS 11.4 (iPhone9,3)   
3    2019-04-03 00:25:28+00:00                          iOS 11.4 (iPhone9,3)   
4    2019-04-03 00:25:29+00:00

In [94]:

type_mapping = []

type_mapping.append({
    "listener_type": "late_night_listener",
    "day_type": "Weekday",
    "day_part": "night",
    "weight": 1.0
})
type_mapping.append({
    "listener_type": "weekwnd_party_listener",
    "day_type": "Weekend",
    "day_part": "night",
    "weight": 1.0
})

# Weekday Focus Day
type_mapping.append({
    "listener_type": "working_focus_listener",
    "day_type": "Weekday",
    "day_part": "morning",
    "weight": 0.5
})
type_mapping.append({
    "listener_type": "working_focus_listener",
    "day_type": "Weekday",
    "day_part": "midday",
    "weight": 0.5
})

# Weekend Focus Day
type_mapping.append({
    "listener_type": "weekend_breakfast_listener",
    "day_type": "Weekend",
    "day_part": "morning",
    "weight": 0.5
})
type_mapping.append({
    "listener_type": "weekend_breakfast_listener",
    "day_type": "Weekend",
    "day_part": "midday",
    "weight": 0.5
})

# Weekday Evening
type_mapping.append({
    "listener_type": "weekday_sundown_listener",
    "day_type": "Weekday",
    "day_part": "evening",
    "weight": 1.0
})

# Weekend Evening
type_mapping.append({
    "listener_type": "weekend_prime_time_listener",
    "day_type": "Weekend",
    "day_part": "evening",
    "weight": 1.0
})

map_df = pd.DataFrame(type_mapping)
merged = pd.merge(grouped, map_df, on=["day_type", "day_part"], how="inner")
merged["weighted_minutes"] = merged["minutes"] * merged["weight"]
result = merged.groupby("listener_type")["weighted_minutes"].sum().reset_index()
result = result.sort_values(by="weighted_minutes", ascending=False).reset_index(drop=True)
total = result["weighted_minutes"].sum()
result["normalized"] = result["weighted_minutes"] / total
print(result)

                 listener_type  weighted_minutes  normalized
0     weekday_sundown_listener         40.168621    0.204292
1   weekend_breakfast_listener         35.915472    0.182661
2       working_focus_listener         32.836068    0.167000
3       weekwnd_party_listener         31.399044    0.159691
4          late_night_listener         31.299491    0.159185
5  weekend_prime_time_listener         25.004801    0.127171


| Type                                  | Description                                                                     | Rule                                                       |
| ------------------------------------- | ------------------------------------------------------------------------------- | ---------------------------------------------------------- |
| 🎧 **late\_night\_listener**          | Mostly listens at night – prefers chill, lo-fi, or introspective music.         | Night listening ≥ 40% of total time                        |
| ☀️ **weekday\_focus\_day\_listener**  | Listens primarily during mornings and midday on weekdays – productive sessions. | Morning + midday ≥ 50%, with more time on weekdays         |
| ☀️ **weekend\_focus\_day\_listener**  | Listens mainly in the morning or midday on weekends – calm and balanced mood.   | Morning + midday ≥ 50%, with more time on weekends         |
| 🌆 **weekday\_prime\_time\_streamer** | Prefers evening listening on weekdays – post-work, gym, or relaxation music.    | Evening is dominant, and weekday evening > weekend evening |
| 🌆 **weekend\_prime\_time\_streamer** | Tunes in most during weekend evenings – party vibes or social listening.        | Evening is dominant, and weekend evening > weekday evening |
| 🎛️ **eclectic\_rhythm\_fan**         | No dominant time pattern – diverse and varied music preferences.                | No strong dominance across time blocks                     |
| 🔇 **silent\_type**                   | No activity or not enough data for classification.                              | Total listening time is 0                                  |
